In [19]:
# Install necessary packages if not already installed
import Pkg
Pkg.add(["CSV", "DataFrames", "Flux", "Statistics", "Random", "Dates", "StatsBase"])

using CSV
using DataFrames
using Flux
using Statistics
using Random
using Dates
using StatsBase

# Include the provided utility functions
# This file contains: buildClassANN, trainClassANN, crossvalidation, ANNCrossValidation, etc.
include("utils.jl")

println("Environment setup complete. Utils loaded.")

   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`


Environment setup complete. Utils loaded.


In [20]:
# Load the dataset
const DATA_PATH = "Fraudulent_E-Commerce_Transaction_Data_merge.csv"

println("Loading data...")
df = CSV.read(DATA_PATH, DataFrame)

# --- SUBSAMPLING (Keep enabled for testing) ---
const DO_SUBSAMPLE = true 
const SUBSAMPLE_SIZE = 10000 

if DO_SUBSAMPLE
    println("\n[Warning] SUBSAMPLING ENABLED: Using only $SUBSAMPLE_SIZE rows.")
    df = df[1:SUBSAMPLE_SIZE, :]
end

# CORRECTED TARGET based on your image
target_col = "Is Fraudulent"

println("Data loaded. Rows: ", size(df, 1))
println("Target Column: ", target_col)
println("Columns found: ", names(df))

Loading data...

⚠️ SUBSAMPLING ENABLED: Using only 10000 rows.
Data loaded. Rows: 10000
Target Column: Is Fraudulent
Columns found: ["Transaction ID", "Customer ID", "Transaction Amount", "Transaction Date", "Payment Method", "Product Category", "Quantity", "Customer Age", "Customer Location", "Device Used", "IP Address", "Shipping Address", "Billing Address", "Is Fraudulent", "Account Age Days", "Transaction Hour"]


In [22]:
function preprocess_data(dataframe)
    data = copy(dataframe)
    println("Preprocessing Avanzato: Gestione Missing, Feature Engineering e One-Hot...")

    # --- 0. Gestione Dati Mancanti (Handling Missing Values) ---
    # Converti gli "?" in missing e gestisci i tipi
    for col in names(data)
        if eltype(data[!, col]) <: AbstractString
            # Tenta di convertire colonne numeriche/date che potrebbero essere state lette come stringhe
            try
                data[!, col] = replace(data[!, col], "?" => missing)
            catch
                # Ignora se la colonna non è di tipo stringa
            end
        end
    end

    # Imputazione Semplice: Sostituisci i missing per colonne chiave
    for col in ["Transaction Amount", "Quantity", "Customer Age", "Account Age Days"]
        if col in names(data) && any(ismissing, data[!, col])
            median_val = median(skipmissing(data[!, col]))
            replace!(data[!, col], missing => median_val)
            println("  - Imputazione Missing: $col (Mediana: $median_val)")
        end
    end

    # --- 1. Feature Engineering (Creazione di nuove features) ---
    
    # a) Estrazione dell'Ora (Mantenuta)
    if !("Transaction Hour" in names(data)) && "Transaction Date" in names(data)
        # Assicurati che 'Transaction Date' sia pulito e possa essere parsato se necessario
        println("  - Estrazione Hour from 'Transaction Date'...")
        # (Logica di estrazione ora omessa qui per brevità, assumendo sia stata fatta o la colonna esista)
        # Nota: L'errore sulla data è stato gestito precedentemente.
    end
    
    # b) Flag di Alto Valore (High-Value Transaction Flag)
    # Crea una feature binaria che segnala se l'importo supera un quantile alto (es. 95th percentile)
    if "Transaction Amount" in names(data)
        threshold_amount = quantile(data[!, "Transaction Amount"], 0.95)
        data.Is_High_Value = [x > threshold_amount for x in data[!, "Transaction Amount"]]
        println("  - Creazione Feature: Is_High_Value (Threshold > $threshold_amount)")
    end

    # c) Flag Account Nuovo/Rischioso
    if "Account Age Days" in names(data)
        # Assumiamo che gli account con meno di 60 giorni siano "nuovi" e potenzialmente più rischiosi
        data.Is_New_Account = [x < 60 for x in data[!, "Account Age Days"]]
        println("  - Creazione Feature: Is_New_Account (Age < 60 days)")
    end


    # --- 2. Pulizia Colonne Non Utili ---
    cols_to_drop = [
        "Transaction ID", 
        "Customer ID", 
        "Transaction Date", 
        "IP Address", 
        "Shipping Address", 
        "Billing Address", 
        "Customer Location" 
    ]
    actual_cols_to_drop = intersect(names(data), cols_to_drop)
    select!(data, Not(actual_cols_to_drop))

    # --- 3. One-Hot Encoding (OHE) per le Colonne Categoriche ---
    categorical_cols = ["Payment Method", "Product Category", "Device Used"]
    df_encoded = data[:, setdiff(names(data), categorical_cols)]
    
    for col in categorical_cols
        if col in names(data)
            encoded_matrix = oneHotEncoding(data[!, col])
            new_col_names = ["$(col)_$(i)" for i in 1:size(encoded_matrix, 2)]
            encoded_df = DataFrame(encoded_matrix, new_col_names)
            df_encoded = hcat(df_encoded, encoded_df)
        end
    end

    # Finalizzazione
    data = df_encoded
    println("\n[OK] Preprocessing completato. Totale Features: ", size(data, 2) - 1)
    return data
end

# Esegui nuovamente la Cella 3 per aggiornare df_processed
println("Preprocessing data...")
df_processed = preprocess_data(df)
println("Remaining Features: ", names(df_processed))

Preprocessing data...
Preprocessing Avanzato: Gestione Missing, Feature Engineering e One-Hot...
  - Creazione Feature: Is_High_Value (Threshold > 671.3819999999996)
  - Creazione Feature: Is_New_Account (Age < 60 days)

✅ Preprocessing completato. Totale Features: 19
Remaining Features: ["Transaction Amount", "Quantity", "Customer Age", "Is Fraudulent", "Account Age Days", "Transaction Hour", "Is_High_Value", "Is_New_Account", "Payment Method_1", "Payment Method_2", "Payment Method_3", "Payment Method_4", "Product Category_1", "Product Category_2", "Product Category_3", "Product Category_4", "Product Category_5", "Device Used_1", "Device Used_2", "Device Used_3"]


In [23]:
# Define the target column again to be safe
target_column = "Is Fraudulent"

# Select input features (all columns except the target)
input_cols = setdiff(names(df_processed), [target_column])

println("Preparing Matrix...")

try
    # 1. Input Matrix (Features)
    global inputs = Matrix{Float32}(df_processed[:, input_cols])
    
    # 2. Target Vector (Labels)
    global targets = df_processed[:, target_column]

    println("[OK] Success!")
    println("Input Matrix Dimensions: ", size(inputs))
    println("Target Vector Dimensions: ", size(targets))
    println("Data Type: ", eltype(inputs))
    
catch e
    println("\n[X] ERROR during Matrix conversion.")
    println("Check the output of Cell 3 again.")
    rethrow(e)
end

Preparing Matrix...
✅ Success!
Input Matrix Dimensions: (10000, 19)
Target Vector Dimensions: (10000,)
Data Type: Float32


In [25]:
# --- Hyperparameters ---

k_folds = 10 
topology = [128, 64, 32] # Topologia più profonda ma non eccessiva
numExecutions = 10 
maxEpochs = 500
learningRate = 0.005 # Ridotto da 0.01

# Uso della funzione di trasferimento Rectified Linear Unit (ReLU)
# Importata dal modulo Flux (già importato in Cella 1)
transferFunctions = fill(relu, length(topology)) # <--- Modifica qui!

validationRatio = 0.2 
maxEpochsVal = 25     

println("Configuration Set:")
println("Topology: Inputs -> $topology -> Outputs")
println("Transfer Function: ReLU")
println("Learning Rate: $learningRate")
println("Folds: $k_folds | Executions per fold: $numExecutions")

Configuration Set:
Topology: Inputs -> [128, 64, 32] -> Outputs
Transfer Function: ReLU
Learning Rate: 0.005
Folds: 10 | Executions per fold: 10


In [26]:
# --- BALANCING THE DATA (OVERSAMPLING) ---
# Since utils.jl does not support weighted loss, we must balance the input data.

println("Balancing dataset via Oversampling...")

# 1. Separate indices
idx_fraud = findall(x -> x == 1, df_processed[!, target_column])
idx_legit = findall(x -> x == 0, df_processed[!, target_column])

# 2. Calculate how many frauds we need to add
num_legit = length(idx_legit)
num_fraud = length(idx_fraud)
num_to_add = num_legit - num_fraud

println("Original: $num_legit Legit, $num_fraud Fraud")

# 3. Randomly sample fraud indices to duplicate
extra_fraud_indices = rand(idx_fraud, num_to_add)

# 4. Create new balanced dataset indices
balanced_indices = vcat(idx_legit, idx_fraud, extra_fraud_indices)
shuffle!(balanced_indices)

# 5. Create the new Inputs/Targets Matrices
# IMPORTANT: We overwrite 'inputs' and 'targets' so the next cell uses the balanced version
global inputs = Matrix{Float32}(df_processed[balanced_indices, input_cols])
global targets = df_processed[balanced_indices, target_column]

# 6. RE-GENERATE CV INDICES for the new balanced data
# We must re-run the crossvalidation function because the data size changed!
global cv_indices = crossvalidation(vec(targets .== 1), k_folds)
global balanced_indices_mapped = balanced_indices
println("[OK] Data Balanced!")
println("New Dataset Size: ", length(targets))
println("New Class Distribution: ", countmap(targets))

Balancing dataset via Oversampling...
Original: 9525 Legit, 475 Fraud
✅ Data Balanced!
New Dataset Size: 19050
New Class Distribution: Dict(0 => 9525, 1 => 9525)


In [27]:
# Generate Stratified Cross-Validation Indices
println("Generating stratified cross-validation indices...")

# FIX: Convert targets to a Boolean Vector (true/false)
# This ensures utils.jl uses the correct binary classification logic
# instead of getting confused and thinking it's a matrix.
targets_bool = vec(targets .== 1)

# Now pass the boolean vector to the function
cv_indices = crossvalidation(targets_bool, k_folds)

println("Indices generated successfully.")
println("First 20 indices: ", cv_indices[1:20])

Generating stratified cross-validation indices...
Indices generated successfully.
First 20 indices: [9, 1, 4, 8, 10, 8, 7, 8, 3, 7, 4, 7, 3, 10, 1, 10, 5, 10, 10, 1]


In [28]:
println("Starting ANN Cross-Validation...")
println("This may take time depending on dataset size and hardware.")

# Call the function from utils.jl 
# Returns a tuple of tuples: ((acc_mean, acc_std), ..., confusionMatrix)
results = ANNCrossValidation(
    topology,
    (inputs, targets),
    cv_indices;
    numExecutions = numExecutions,
    transferFunctions = transferFunctions,
    maxEpochs = maxEpochs,
    learningRate = learningRate,
    validationRatio = validationRatio,
    maxEpochsVal = maxEpochsVal
)

println("Training Complete!")

Starting ANN Cross-Validation...
This may take time depending on dataset size and hardware.

Fold 1/10
  Execution 10/10

Fold 2/10
  Execution 10/10

Fold 3/10
  Execution 10/10

Fold 4/10
  Execution 10/10

Fold 5/10
  Execution 10/10

Fold 6/10
  Execution 10/10

Fold 7/10
  Execution 10/10

Fold 8/10
  Execution 10/10

Fold 9/10
  Execution 10/10

Fold 10/10
  Execution 10/10
Training Complete!


In [29]:
# Unpack the results [cite: 64]
(acc, err, sens, spec, ppv, npv, f1, global_cm) = results

println("=== Cross-Validation Results (Mean ± STD) ===")
println("Accuracy:    $(round(acc[1]*100, digits=2))% ± $(round(acc[2]*100, digits=2))%")
println("Error Rate:  $(round(err[1]*100, digits=2))% ± $(round(err[2]*100, digits=2))%")
println("Sensitivity: $(round(sens[1]*100, digits=2))% ± $(round(sens[2]*100, digits=2))%")
println("Specificity: $(round(spec[1]*100, digits=2))% ± $(round(spec[2]*100, digits=2))%")
println("Precision (PPV): $(round(ppv[1]*100, digits=2))% ± $(round(ppv[2]*100, digits=2))%")
println("F1 Score:    $(round(f1[1]*100, digits=2))% ± $(round(f1[2]*100, digits=2))%")

println("\n=== Global Confusion Matrix (Summed & Averaged) ===")
# Note: Values are floats because they are averages of multiple executions
display(global_cm)

# Interpretation Helper
println("\n--- Matrix Interpretation ---")
println("Row 1: Actual Non-Fraud")
println("Row 2: Actual Fraud")
println("Col 1: Predicted Non-Fraud")
println("Col 2: Predicted Fraud")

TP = global_cm[2,2]
FP = global_cm[1,2]
FN = global_cm[2,1]

println("\nFraud Detection Analysis:")
println("We successfully caught approx $(round(TP, digits=1)) fraud cases.")
println("We missed approx $(round(FN, digits=1)) fraud cases.")
println("We falsely accused approx $(round(FP, digits=1)) legitimate users.")

=== Cross-Validation Results (Mean ± STD) ===
Accuracy:    95.97% ± 0.35%
Error Rate:  4.03% ± 0.35%
Sensitivity: 92.38% ± 0.71%
Specificity: 99.57% ± 0.28%
Precision (PPV): 99.53% ± 0.3%
F1 Score:    95.82% ± 0.37%

=== Global Confusion Matrix (Summed & Averaged) ===


2×2 Matrix{Float64}:
 9483.6    41.4
  725.7  8799.3


--- Matrix Interpretation ---
Row 1: Actual Non-Fraud
Row 2: Actual Fraud
Col 1: Predicted Non-Fraud
Col 2: Predicted Fraud

Fraud Detection Analysis:
We successfully caught approx 8799.3 fraud cases.
We missed approx 725.7 fraud cases.
We falsely accused approx 41.4 legitimate users.


In [30]:
# --- CELLA 9: Analisi del Livello di Rischio (Risk Scoring) ---

println("Generazione del Profilo di Rischio...")

# 1. Addestriamo un modello finale su un split 80/20 (Hold-Out)
# Usiamo la topologia ottimizzata [64, 32, 16]
(train_idx, val_idx, test_idx) = holdOut(size(inputs, 1); Pval=0.0, Ptest=0.2)

# Preparazione del set di Training
train_inputs = inputs[train_idx, :]
# Correzione 1: Convertiamo i target a Bool e rimodelliamo in una matrice a 1 colonna
train_targets_bool_matrix = reshape(targets[train_idx] .== 1, :, 1)

# Normalizzazione del Training Set (necessaria per il modello finale)
normParams_final = calculateMinMaxNormalizationParameters(train_inputs)
train_inputs_norm = normalizeMinMax(train_inputs, normParams_final)

# Correzione 2: Rimuoviamo il parametro printLoss e usiamo i nuovi parametri
(ann_final, _) = trainClassANN(topology, 
    (train_inputs_norm, train_targets_bool_matrix);
    transferFunctions=transferFunctions, # Usiamo la ReLU
    maxEpochs=maxEpochs, 
    learningRate=learningRate 
)

# 2. Otteniamo le probabilità pure (valori tra 0 e 1) sul set di test
test_inputs = inputs[test_idx, :]
# Normalizzazione del Test Set usando i parametri del Training
test_inputs_norm = normalizeMinMax(test_inputs, normParams_final)

# Prediction: Trasponiamo gli inputs per Flux (features x samples)
test_probs = ann_final(test_inputs_norm')' 
test_probs = vec(test_probs) # Estraiamo la singola colonna di probabilità

# 3. Categorizziamo il rischio in fasce
risk_levels = String[]
for p in test_probs
    if p < 0.2
        push!(risk_levels, "1. Basso Rischio")
    elseif p < 0.6
        push!(risk_levels, "2. Rischio Moderato")
    elseif p < 0.9
        push!(risk_levels, "3. Alto Rischio")
    else
        push!(risk_levels, "4. Rischio Critico (Frode Certa)")
    end
end

# 4. Visualizziamo la distribuzione
println("\n--- Distribuzione dei Livelli di Rischio (Test Set) ---")
risk_counts = countmap(risk_levels)
# Ordiniamo per chiave per una lettura pulita
for level in sort(collect(keys(risk_counts)))
    count = risk_counts[level]
    perc = round(count / length(test_probs) * 100, digits=1)
    println("$level: $count transazioni ($perc%)")
end
# ... (Fine della Cella 20 dopo la stampa della Distribuzione) ...

# 5. Valutazione dell'Attendibilità delle Fasce Alto/Critico Rischio

# Target Reale del Test Set (usiamo il vettore booleano per l'analisi)
test_targets_bool = targets[test_idx] .== 1

# Predizioni "Allarme": Vero se la probabilità è >= 0.6 (Alto o Critico Rischio)
allarme_predetto = test_probs .>= 0.6 

# Calcolo delle metriche sulla Fascia di Rischio

TP = sum(allarme_predetto .& test_targets_bool)    # Frodi che il modello ha "allarmato"
FP = sum(allarme_predetto .& .!test_targets_bool)  # Non-frodi che il modello ha "allarmato" (Falsi Allarmi)
FN = sum(.!allarme_predetto .& test_targets_bool) # Frodi mancate (Finite in Basso/Moderato)
TN = sum(.!allarme_predetto .& .!test_targets_bool) # Non-frodi correttamente non allarmate

# Precisione (PPV) per l'Allarme: Quanta fiducia possiamo avere nell'allarme?
precisione_allarme = (TP + FP) > 0 ? TP / (TP + FP) : 0.0

# Recall (Sensibilità): Quante frodi abbiamo coperto con la fascia di allarme?
recall_allarme = (TP + FN) > 0 ? TP / (TP + FN) : 0.0

println("\n--- Valutazione Fascia di Allarme (Rischio Alto/Critico > 0.6) ---")
println("Precisione (PPV) dell'Allarme: $(round(precisione_allarme*100, digits=2))%")
println("Recall (Sensibilità) dell'Allarme: $(round(recall_allarme*100, digits=2))%")
println("Falsi Allarmi (FP): $FP")
println("Frodi Mancate (FN): $FN")

Generazione del Profilo di Rischio...

--- Distribuzione dei Livelli di Rischio (Test Set) ---
1. Basso Rischio: 1682 transazioni (44.1%)
2. Rischio Moderato: 102 transazioni (2.7%)
3. Alto Rischio: 363 transazioni (9.5%)
4. Rischio Critico (Frode Certa): 1663 transazioni (43.6%)

--- Valutazione Fascia di Allarme (Rischio Alto/Critico > 0.6) ---
Precisione (PPV) dell'Allarme: 93.98%
Recall (Sensibilità) dell'Allarme: 99.9%
Falsi Allarmi (FP): 122
Frodi Mancate (FN): 2


In [35]:
# --- CELLA 23: Calcolo Permutation Importance (F1-Score) ---

using DataFrames
using Statistics
using Random # Necessario per shuffle!

println("Calcolo Permutation Importance basato su F1-Score...")

# 1. Preparazione del Test Set (usando la stessa logica di Cella 20)
test_inputs = inputs[test_idx, :]
test_targets_bool_matrix = reshape(targets[test_idx] .== 1, :, 1)
test_inputs_norm = normalizeMinMax(test_inputs, normParams_final)
num_features = size(test_inputs_norm, 2)
feature_names = input_cols # Nomi delle colonne OHE dalla Cella 4

# Funzione per calcolare l'F1-Score del Test Set
function calculate_f1(ann, inputs_norm, targets)
    # Flux richiede inputs (features x samples)
    test_outputs = ann(inputs_norm')'
    # F1-Score è il 7° elemento restituito da confusionMatrix (funzione da utils.jl)
    (_, _, _, _, _, _, f1, _) = confusionMatrix(test_outputs, targets)
    return f1
end

# 2. Calcolo del Baseline F1-Score
baseline_f1 = calculate_f1(ann_final, test_inputs_norm, test_targets_bool_matrix)
println("Baseline F1-Score: $(round(baseline_f1*100, digits=2))%")

# 3. Permutation Loop
importance_scores = Dict{String, Float64}()

# Copia del Test Set Normalizzato per la permutazione
inputs_to_permute = copy(test_inputs_norm) 

for i in 1:num_features
    feature_name = feature_names[i]
    
    # 3a. Shuffling (Permutazione): Mescola solo la colonna i
    shuffle!(view(inputs_to_permute, :, i))
    
    # 3b. Calcola l'F1-Score con la feature mescolata
    shuffled_f1 = calculate_f1(ann_final, inputs_to_permute, test_targets_bool_matrix)
    
    # 3c. Calcola l'Importanza: Drop nella performance
    drop_in_f1 = baseline_f1 - shuffled_f1
    importance_scores[feature_name] = drop_in_f1
    
    # 3d. Ripristina i dati originali per la prossima iterazione
    inputs_to_permute = copy(test_inputs_norm) 
end

# 4. Classificazione e Output
sorted_importance = sort(collect(importance_scores), by=x->x[2], rev=true)

println("\n=== TOP 5 Feature Importance (Permutation F1-Score Drop) ===")
for (i, (feature, drop)) in enumerate(sorted_importance)
    if i > 5
        break
    end
    # Convertiamo i valori a punti percentuali per una migliore lettura
    drop_percent = round(drop * 100, digits=2)
    println("$(i). $(feature): Calo di F1 del $(drop_percent) pp (punti percentuali)")
end

# Salviamo la classifica completa in un DataFrame per riferimento futuro
importance_df = DataFrame(
    Feature = [x[1] for x in sorted_importance], 
    Importance_Drop_F1 = [x[2] for x in sorted_importance]
)
CSV.write("feature_importance.csv", importance_df)
println("\nClassifica completa salvata in feature_importance.csv")

Calcolo Permutation Importance basato su F1-Score...
Baseline F1-Score: 96.33%

=== TOP 5 Feature Importance (Permutation F1-Score Drop) ===
1. Account Age Days: Calo di F1 del 37.82 pp (punti percentuali)
2. Transaction Hour: Calo di F1 del 26.74 pp (punti percentuali)
3. Quantity: Calo di F1 del 23.95 pp (punti percentuali)
4. Is_New_Account: Calo di F1 del 20.15 pp (punti percentuali)
5. Transaction Amount: Calo di F1 del 18.96 pp (punti percentuali)

Classifica completa salvata in feature_importance.csv


In [36]:
# --- CELLA 10: Ranking Frodi per Frequenza e Costo (Integrato con Modello) ---

using DataFrames

println("\n--- RANKING FRODI: Analisi delle Predizioni ad Alto Rischio ---")

# 1. Mappatura degli Indici: Troviamo quali righe originali (1-10000) corrispondono al Test Set (3810 indici)
# 'test_idx' è un indice del set bilanciato (19050 righe)
# 'balanced_indices_mapped' ci dice a quale riga originale (1-10000) corrisponde ogni riga bilanciata.

# Otteniamo gli indici originali che compongono il set di test
original_indices_in_test_set = balanced_indices_mapped[test_idx] 

# 2. Creiamo un DataFrame di Analisi UNIFICATO dal DF Originale (df)
# Usiamo gli indici mappati sul DataFrame originale (df ha solo 10000 righe)
df_analysis = df[original_indices_in_test_set, [
    "Product Category", 
    "Device Used", 
    "Transaction Amount", 
    "Is Fraudulent" 
]]

# 3. Aggiungiamo i Livelli di Rischio PRECISI generati dal MODELLO (Cella 20)
# 'risk_levels' ha la stessa dimensione del set di test (3810)
df_analysis.Risk_Level = risk_levels

# Filtriamo solo le transazioni che il MODELLO ha etichettato come ALTO/CRITICO RISCHIO
high_risk_data = filter(row -> startswith(row.Risk_Level, "3.") || startswith(row.Risk_Level, "4."), df_analysis)


# 4. Ranking per CATEGORIA DI PRODOTTO (nelle transazioni ad alto rischio)
println("\nTOP 5 Categorie (per Totale Perdita) nelle Transazioni ad Alto/Critico Rischio (Secondo il Modello):")

cat_stats = combine(groupby(high_risk_data, "Product Category"), 
                    nrow => :Count, 
                    "Transaction Amount" => sum => :Total_Loss)

sort!(cat_stats, :Total_Loss, rev=true)
display(first(cat_stats, 5))


# 5. Ranking per DEVICE (Dispositivo) (nelle transazioni ad alto rischio)
println("\nTOP Device (per Frequenza) nelle Transazioni ad Alto/Critico Rischio (Secondo il Modello):")

dev_stats = combine(groupby(high_risk_data, "Device Used"), 
                    nrow => :Count,
                    "Transaction Amount" => mean => :Avg_Loss)

sort!(dev_stats, :Count, rev=true)
display(dev_stats)


println("\nInsight: L'analisi ora rispetta la predizione del modello, concentrandosi sui segmenti che la Rete Neurale ha identificato come problematici.")


--- RANKING FRODI: Analisi delle Predizioni ad Alto Rischio ---

TOP 5 Categorie (per Totale Perdita) nelle Transazioni ad Alto/Critico Rischio (Secondo il Modello):


Row,Product Category,Count,Total_Loss
,String15,Int64,Float64
1,health & beauty,489,2.71252e5
2,toys & games,407,2.46788e5
3,home & garden,397,222825.0
4,electronics,340,2.07049e5
5,clothing,393,1.96303e5



TOP Device (per Frequenza) nelle Transazioni ad Alto/Critico Rischio (Secondo il Modello):


Row,Device Used,Count,Avg_Loss
,String7,Int64,Float64
1,tablet,704,532.721
2,desktop,678,573.196
3,mobile,644,590.924



Insight: L'analisi ora rispetta la predizione del modello, concentrandosi sui segmenti che la Rete Neurale ha identificato come problematici.


In [37]:
# --- CELLA 22: Analisi Comportamentale Anomala (Integrata con Modello) ---

using DataFrames
using Statistics

println("\n--- ANALISI COMPORTAMENTO ANOMALO (basata sulle Predizioni ANN) ---")

# 1. Preparazione del DataFrame di Analisi (Unione di df con i risultati del modello)
# Usiamo gli indici mappati (calcolati in Cella 21) per recuperare le colonne descrittive originali
df_analysis = df[original_indices_in_test_set, [
    "Transaction Amount", 
    "Transaction Hour", 
    "Is Fraudulent" # Il target reale per il confronto
]]

# Aggiungiamo i Livelli di Rischio generati dal MODELLO
df_analysis.Risk_Level = risk_levels

# Filtriamo solo le transazioni che il MODELLO ha etichettato come ALTO/CRITICO RISCHIO
high_risk_data = filter(row -> startswith(row.Risk_Level, "3.") || startswith(row.Risk_Level, "4."), df_analysis)
num_high_risk = nrow(high_risk_data)

# --- 2. Analisi Temporale: A che ora il Modello Allarma Maggiormente? ---

# Raggruppiamo gli allarmi per "Transaction Hour"
hour_stats = combine(groupby(high_risk_data, "Transaction Hour"), nrow => :Count)
sort!(hour_stats, :Count, rev=true)

println("\nOrari più pericolosi (Top 3 ore con più ALLARMI del Modello):")
for row in eachrow(first(hour_stats, 3))
    println("  Ore $(row["Transaction Hour"]): $(row[:Count]) allarmi rilevati")
end


# --- 3. Anomalie sugli Importi (High Value Whales negli Allarmi) ---

# Importo medio di tutte le transazioni (stima del legittimo - usiamo il DF iniziale per la media generale)
mean_val_original = mean(df[:, "Transaction Amount"]) 
# Importo medio delle transazioni che il MODELLO ha allarmato
alarm_mean = mean(high_risk_data[:, "Transaction Amount"])

println("\nConfronto Importi Medi (Modello vs. Media Generale):")
println("  Importo medio transazione generale (stima): $(round(mean_val_original, digits=2))")
println("  Importo medio transazione allarmata:         $(round(alarm_mean, digits=2))")
println("  Totale transazioni allarmate analizzate:     $num_high_risk")

if alarm_mean > mean_val_original
    println(">> CONCLUSIONE: Le transazioni allarmate dall'ANN tendono ad avere importi PIÙ ALTI della media.")
else
    println(">> CONCLUSIONE: Le transazioni allarmate dall'ANN tendono ad essere piccoli importi.")
end


--- ANALISI COMPORTAMENTO ANOMALO (basata sulle Predizioni ANN) ---

Orari più pericolosi (Top 3 ore con più ALLARMI del Modello):
  Ore 1: 209 allarmi rilevati
  Ore 5: 189 allarmi rilevati
  Ore 2: 184 allarmi rilevati

Confronto Importi Medi (Modello vs. Media Generale):
  Importo medio transazione generale (stima): 226.09
  Importo medio transazione allarmata:         564.77
  Totale transazioni allarmate analizzate:     2026
>> CONCLUSIONE: Le transazioni allarmate dall'ANN tendono ad avere importi PIÙ ALTI della media.


In [38]:
# --- CELLA 24: Ottimizzazione del Threshold tramite Curva Precision-Recall ---

using Statistics
using DataFrames

println("Ottimizzazione del Threshold per massimizzare la Recall e la Precisione...")

# 1. Preparazione dei dati (Assunti dalla Cella 20)
# test_probs: Vettore di probabilità [0, 1] generate dal modello ANN sul Test Set.
# test_targets_bool: Vettore Bool dei target reali del Test Set.

# Se test_targets_bool non è globale, lo definiamo qui
# test_targets_bool = targets[test_idx] .== 1 

# 2. Definizione dei Threshold da testare
thresholds = 0.05:0.01:0.95 # Da 5% a 95% con incrementi dell'1%

metrics = DataFrame(Threshold=Float64[], Precision=Float64[], Recall=Float64[], F1_Score=Float64[])

# 3. Loop per calcolare Precisione e Recall per ogni Threshold
for t in thresholds
    predictions = test_probs .>= t
    targets_real = test_targets_bool

    # Calcolo TP, FP, FN (Necessari per Precisione e Recall)
    TP = sum(predictions .& targets_real)
    FP = sum(predictions .& .!targets_real)
    FN = sum(.!predictions .& targets_real)
    
    precision = (TP + FP) > 0 ? TP / (TP + FP) : 1.0 # Precisione (PPV)
    recall = (TP + FN) > 0 ? TP / (TP + FN) : 0.0   # Recall (Sensibilità)
    
    f1 = (precision + recall) > 0 ? 2 * precision * recall / (precision + recall) : 0.0
    
    push!(metrics, (t, precision, recall, f1))
end


# 4. Trovare il Threshold Ottimale per l'Obiettivo
# Obiettivo 1: Massimizzare l'F1-Score
best_f1_row = metrics[argmax(metrics.F1_Score), :]

# Obiettivo 2: Ottenere una Recall molto alta mantenendo una Precisione minima accettabile (es. > 85%)
min_acceptable_precision = 0.85
filtered_metrics = filter(row -> row.Precision >= min_acceptable_precision, metrics)

# Se ci sono righe valide, prendi quella con la Recall più alta
if nrow(filtered_metrics) > 0
    best_recall_row = filtered_metrics[argmax(filtered_metrics.Recall), :]
    optimal_threshold = best_recall_row.Threshold
else
    # Fallback all'F1 se la Precisione minima non è raggiunta
    optimal_threshold = best_f1_row.Threshold
end


println("\n--- Risultati Ottimizzazione Threshold ---")
println("Threshold Ottimale (Massima Recall con Precisione > 85%): $(round(optimal_threshold, digits=2))")
println("Metrica F1 Massima (per confronto): $(round(best_f1_row.Threshold, digits=2))")
println("\nMetrica scelta per il Threshold $(round(optimal_threshold, digits=2)):")
println("  - Precisione (PPV): $(round(metrics[metrics.Threshold .== optimal_threshold, :Precision][1]*100, digits=2))%")
println("  - Recall (Sensibilità): $(round(metrics[metrics.Threshold .== optimal_threshold, :Recall][1]*100, digits=2))%")

Ottimizzazione del Threshold per massimizzare la Recall e la Precisione...

--- Risultati Ottimizzazione Threshold ---
Threshold Ottimale (Massima Recall con Precisione > 85%): 0.05
Metrica F1 Massima (per confronto): 0.73

Metrica scelta per il Threshold 0.05:
  - Precisione (PPV): 85.78%
  - Recall (Sensibilità): 100.0%


In [39]:
# --- CELLA 25: Analisi di Coorte e Geospaziale (Basata sul Modello ANN) ---

using DataFrames
using Statistics

println("\n--- ANALISI FOCALIZZATA SUL MODELLO: Coorte e Anomalia Geografica ---")

# 1. Creazione del DataFrame di Analisi (Test Set con Predizioni)
# Usiamo gli indici mappati per recuperare le colonne descrittive originali (df)
df_analysis = df[original_indices_in_test_set, [
    "Account Age Days", 
    "Shipping Address", 
    "Billing Address",
    "Is Fraudulent" # Il target reale
]]

# Aggiungiamo i Livelli di Rischio generati dal MODELLO
df_analysis.Risk_Level = risk_levels

# Definiamo la condizione di Allarme (Rischio Alto o Critico)
df_analysis.Is_Alarmed = [startswith(r, "3.") || startswith(r, "4.") for r in df_analysis.Risk_Level]

# ====================================================================
# 1. ANALISI DI COORTE: Frequenza di Allarme per Anzianità del Cliente
# ====================================================================

println("\n=== 1. Frequenza di Allarme del Modello per Anzianità dell'Account ===")

# Funzione per definire la Coorte
function assign_cohort(days::Real)
    # Assicurati che days sia un numero gestendo floating points o tipi misti
    days_int = Int(round(days))
    if days_int <= 30
        return "1. Nuovi Clienti (0-30gg)"
    elseif days_int <= 90
        return "2. Giovani Account (31-90gg)"
    else
        return "3. Account Stabili (91+gg)"
    end
end

df_analysis.Account_Cohort = [assign_cohort(days) for days in df_analysis[!,"Account Age Days"]]

# Aggregazione: Calcola la Percentuale di Allarmi in ogni Coorte
cohort_stats = combine(groupby(df_analysis, :Account_Cohort),
                       nrow => :Total_Transactions_in_Test,
                       :Is_Alarmed => mean => :Model_Alarm_Rate) # mean di Bool è il tasso di True

sort!(cohort_stats, :Model_Alarm_Rate, rev=true)
display(cohort_stats)

println("\nInsight Coorte: I tassi più alti indicano i segmenti che la Rete Neurale considera più a rischio.")


--- ANALISI FOCALIZZATA SUL MODELLO: Coorte e Anomalia Geografica ---

=== 1. Frequenza di Allarme del Modello per Anzianità dell'Account ===


Row,Account_Cohort,Total_Transactions_in_Test,Model_Alarm_Rate
,String,Int64,Float64
1,1. Nuovi Clienti (0-30gg),1018,0.888016
2,2. Giovani Account (31-90gg),533,0.412758
3,3. Account Stabili (91+gg),2259,0.399292



Insight Coorte: I tassi più alti indicano i segmenti che la Rete Neurale considera più a rischio.
